In [35]:

import torch, torchvision
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import DataLoader
import time
import os
import cv2
from dataPreparation import dataPrep
from imutils import paths
from sklearn.model_selection import train_test_split
from torch.nn import Module
from FCN import FCN
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from tqdm import tqdm
from earlyStopping import EarlyStopping



In [3]:

dataset_path = "/Users/beyzaecemerce/Desktop/GitHub/thesis/drone/dataset/semantic_drone_dataset/"
image_path=dataset_path+'original_images'
masked_path = dataset_path+ 'label_images_semantic'
TEST_SPLIT = 0.20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = True if DEVICE == "cuda" else False

In [20]:
image = cv2.imread(image_path+"/100.jpg")
mask=cv2.imread(masked_path+"/100.png")
print(image.shape)
print(mask.shape)
figure, ax = plt.subplots(nrows=1, ncols=2, figsize=(10, 10))
ax[0].grid(False)
ax[1].grid(False)
ax[0].imshow(image)
ax[1].imshow(mask)
ax[0].set_title("Image")
ax[1].set_title("Original Mask")
figure.tight_layout()
figure.show()

(4000, 6000, 3)
(4000, 6000, 3)


/var/folders/xy/y85s_w552q1_8cd2gzzrqfmc0000gn/T/ipykernel_12560/754042599.py:13: UserWarning: Matplotlib is currently using module://matplotlib_inline.backend_inline, which is a non-GUI backend, so cannot show the figure.
  figure.show()


In [3]:
mask=cv2.imread(masked_path+"/100.png")

print(mask.shape)

(4000, 6000, 3)


In [25]:
path="/Users/beyzaecemerce/Desktop/GitHub/thesis/drone/dataset/semantic_drone_dataset/label_images_semantic/001.png"
mask=cv2.imread(path)
print(mask.shape)
print(np.unique(mask))
print(mask[50,500])

mask=np.max(mask,axis=2)
print(mask.shape)
print(mask[50,500])

(4000, 6000, 3)
[ 0  1  2  3  8  9 10 11 12 13 14 15 19 20 21 22]
[22 22 22]
(4000, 6000)
22


In [11]:
path="/Users/beyzaecemerce/Desktop/GitHub/thesis/drone/dataset/semantic_drone_dataset/label_images_semantic/001.png"
mask=cv2.imread(path)
mask=torch.from_numpy(mask)
print(mask.shape)
x,y=torch.max(mask,dim=2)
print(x)
print(y)


torch.Size([4000, 6000, 3])
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 9,  ..., 0, 0, 0],
        [0, 0, 9,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]], dtype=torch.uint8)
tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])


In [22]:
type(mask)

numpy.ndarray

In [15]:
mask.shape

(1024, 2048)

In [26]:
h, w = mask.shape
target = torch.zeros(24, h, w)
for c in range(24):
    target[c][mask == c] = 1

In [28]:
target.shape

torch.Size([24, 4000, 6000])

In [27]:
print(target[:,50,500])

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 1., 0.])


In [12]:
INIT_LR = 2e-4
NUM_EPOCHS = 300
BATCH_SIZE = 32

INPUT_IMAGE_WIDTH = 6000
INPUT_IMAGE_HEIGHT = 4000

BASE_OUTPUT ="/Users/beyzaecemerce/Desktop/GitHub/thesis/drone/output"

MODEL_PATH = os.path.join(BASE_OUTPUT, "MODEL.pth")
PLOT_PATH = os.path.sep.join([BASE_OUTPUT, "Train_Test_Plot.png"])
TEST_PATHS = os.path.sep.join([BASE_OUTPUT, "test_path.txt"])

In [13]:
imagePaths = sorted(list(paths.list_images(image_path)))
maskPaths = sorted(list(paths.list_images(masked_path)))

split = train_test_split(imagePaths, maskPaths,test_size=TEST_SPLIT, random_state=42)

(trainImages, testImages) = split[:2]
(trainMasks, testMasks) = split[2:]

print("[INFO] saving testing image paths...")
f = open(TEST_PATHS, "w")
f.write("\n".join(testImages))
f.close()

[INFO] saving testing image paths...


In [32]:
input_transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((384, 512),interpolation=cv2.INTER_NEAREST),  # Resize the input image to (height=256, width=384)
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
mask_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((384, 512)),  # Resize the mask to the same size as the input image
    transforms.ToTensor()
])

In [36]:
train = dataPrep(imagePaths=trainImages, maskPaths=trainMasks, input_transform=input_transform, mask_transform=mask_transform,n_class=24)
test = dataPrep(imagePaths=testImages, maskPaths=testMasks, input_transform=input_transform, mask_transform=mask_transform,n_class=24)


trainLoader = DataLoader(train, shuffle=False,batch_size=BATCH_SIZE, pin_memory=PIN_MEMORY,num_workers=os.cpu_count())

testLoader = DataLoader(test, shuffle=False,batch_size=BATCH_SIZE, pin_memory=PIN_MEMORY,num_workers=os.cpu_count())


In [37]:
for (i, (x, y)) in enumerate(trainLoader):
    if i==1:
        (x, y) = (x.to(DEVICE), y.to(DEVICE))
        print(x.shape)
        print(y.shape)

torch.Size([32, 3, 384, 512])
torch.Size([32, 24, 384, 512])


In [13]:
model=FCN(20)
opt = Adam(model.parameters(), lr=INIT_LR)
lossFunc=torch.nn.CrossEntropyLoss()
trainSteps = len(train) // BATCH_SIZE
testSteps = len(test) // BATCH_SIZE
H = {"train_loss": [], "test_loss": []}


In [14]:
print("[INFO] training the network...")
startTime = time.time()
for e in tqdm(range(NUM_EPOCHS)):
  model.train()
  totalTrainLoss = 0
  totalTestLoss = 0

  for (i, (x, y)) in enumerate(trainLoader):
    (x, y) = (x.to(DEVICE), y.to(DEVICE))

    print("x",x.shape)
    print("y",y.shape)
    pred = model(x)
    print(pred.shape)
    #pred = torch.argmax(pred, dim=1)
    print(y.shape)
    loss = lossFunc(pred, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    totalTrainLoss += loss

  with torch.no_grad():
    model.eval()
    for (x, y) in testLoader:
      (x, y) = (x.to(DEVICE), y.to(DEVICE))
      pred = model(x)
      totalTestLoss += lossFunc(pred, y)
  avgTrainLoss = totalTrainLoss / trainSteps
  avgTestLoss = totalTestLoss / testSteps
  H["train_loss"].append(avgTrainLoss.cpu().detach().numpy())
  H["test_loss"].append(avgTestLoss.cpu().detach().numpy())
  early_stopping(avgTrainLoss,avgTestLoss)
        
  if early_stopping.early_stop:
      print("Early stopping")
      break
  print("[INFO] EPOCH: {}/{}".format(e + 1, NUM_EPOCHS))
  print("Train loss: {:.6f}, Test loss: {:.4f}".format(avgTrainLoss, avgTestLoss))
endTime = time.time()
print("[INFO] total time taken to train the model: {:.2f}s".format(endTime - startTime))

[INFO] training the network...


  0%|          | 0/300 [00:02<?, ?it/s]


ValueError: Caught ValueError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/_utils/worker.py", line 308, in _worker_loop
    data = fetcher.fetch(index)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 51, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/_utils/fetch.py", line 51, in <listcomp>
    data = [self.dataset[idx] for idx in possibly_batched_index]
  File "/Users/beyzaecemerce/Desktop/GitHub/thesis/dataPreparation.py", line 37, in __getitem__
    h, w = mask.size()
ValueError: too many values to unpack (expected 2)


In [ ]:
h, w = label.size()
target = torch.zeros(self.n_class, h, w)
for c in range(self.n_class):
    target[c][label == c] = 1